# Reasoning SLM - Stage 1: Eval Harness and Baselines

## Introduction

This project takes **Qwen2.5-0.5B** (base) through supervised fine-tuning (SFT) and then GRPO on **GSM8K**, and measures what each stage adds against Qwen's own **Qwen2.5-0.5B-Instruct**, which is used for evaluation only and never trained. Each stage lives in its own notebook:

| Stage | Notebook |
| --- | --- |
| Stage 1: Eval harness and baselines | `01_baseline.ipynb` (this notebook) |
| Stage 2: LoRA SFT | `02_sft.ipynb` |
| Stage 3: GRPO on top of SFT | `03_grpo.ipynb` |

This notebook builds the measuring tool before training anything, then scores the two Qwen models:

- An answer extractor, written once and shared by every stage.
- A fixed, seeded 200-problem validation subset for iterating.
- Fixed eval settings for every model: greedy decoding, at most 512 new tokens, the same stop rules.
- The base model with an 8-shot prompt (and zero-shot for comparison), the instruct model zero-shot with its chat template.
- A 20-prompt general check (non-math instructions), judged pass/fail by hand, to measure what specialization costs in later stages.

**Answer format.** Reasoning steps, then a final line `#### <number>`, the format GSM8K already uses. The same format is used for the SFT targets, the GRPO reward and eval extraction.

## Setup

Clone the repository so the shared code in `src/` can be imported, and install the requirements. Results are written to `results/stage1/`. Download them after each run, because `/content` is wiped when the runtime disconnects.

In [ ]:
import os

REPO_URL = "https://github.com/JuaniLlaberia/slm-post-training.git"
REPO_DIR = "/content/slm-post-training"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

### Imports

In [ ]:
from src.config import BASE_MODEL, INSTRUCT_MODEL, RESULTS_DIR
from src.data.gsm8k import load_gsm8k
from src.evaluation.extraction import lenient_extractor
from src.evaluation.general_check import GENERAL
from src.evaluation.generation import generate
from src.evaluation.harness import evaluate, save_results
from src.prompts.few_shot import EXAMPLES, STOP, build_instruct_prompt, build_prompt

STAGE_DIR = RESULTS_DIR / "stage1"

## Data

GSM8K (`openai/gsm8k`, config `main`) has 7,473 train and 1,319 test problems. A seeded 200-problem subset is held out from train as the validation set used for iterating; the full 1,319-problem test set is kept for final numbers.

`load_gsm8k` also adds a `gold` column (the number after `####`, as an integer) and strips GSM8K's calculator annotations (`<<48/2=24>>`) from the solutions.

In [ ]:
train_dataset, val_dataset, test_dataset = load_gsm8k()

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 7273
Validation dataset size: 200
Test dataset size: 1319


## Eval Harness

### Answer Extractor

`answer_extractor` takes the number after the last `####`, falling back to the last `\boxed{...}` (Qwen instruct's habit). Commas are removed and the result is compared to the gold as a number, not as a string. An output with no answer found counts as wrong, and its rate is logged: a high no-answer rate points to a format problem, not a math problem.

### Prompts

- **Base model:** a few-shot prompt with 8 solved train examples in the `####` format (`EXAMPLES`), each written as `Question: … Answer: …`, followed by the new question. Generation stops at `\nQuestion:` so the model doesn't continue with a new problem.
- **Instruct model:** its own chat template, zero-shot, with an instruction to solve step by step and end with a final line `#### <number>`.

### Generation and Scoring

`generate` runs greedy decoding in batches (at most 512 new tokens, no repetition penalty) and cuts each output at the first stop string. `evaluate` scores every output and prints the accuracy, the no-answer rate and the share of outputs that hit the token limit. Every generation is saved to JSON with its question, output, extracted answer, gold answer, correctness, token count and whether it hit the limit.

## Base Model Baseline

### 8-shot

In [ ]:
questions = val_dataset["question"]
golds = val_dataset["gold"]

results = evaluate(BASE_MODEL, lambda q, _: build_prompt(q, examples=EXAMPLES), questions, golds, stop=STOP)

Qwen/Qwen2.5-0.5B: acc=48.0%  no-answer=4.0%


In [ ]:
save_results(results, STAGE_DIR / "base_8shot_val200.json")

With the same 8 examples as the few-shot prompt for every question, `Qwen2.5-0.5B` reaches **48%** accuracy on the 200 validation problems, with a **4%** no-answer rate.

### Zero-shot

In [ ]:
results = evaluate(BASE_MODEL, lambda q, _: build_prompt(q, examples=[]), questions, golds, stop=STOP)

Qwen/Qwen2.5-0.5B: acc=0.0%  no-answer=96.0%  hit-limit=54.5%


In [ ]:
save_results(results, STAGE_DIR / "base_0shot_val200.json")

In [ ]:
hidden = sum(lenient_extractor(r["output"]) == r["gold"] for r in results)
print(f"correct but unformatted: {hidden}/{len(results)}")

correct but unformatted: 97/200


Without examples, Qwen2.5-0.5B base scores **0%** on the 200 validation problems: 96% of outputs contain no `####` line, and 54.5% run until the 512-token limit. The model isn't failing at math. It answers in its own words ("The answer is 16") and then keeps generating. A lenient check, which reads the first "answer is N" in each output, finds **97/200 (48.5%)** correct but unformatted answers. Zero-shot, the base model knows the answers but doesn't know the expected output format or when to stop.

## Instruct Model Baseline

In [ ]:
results = evaluate(INSTRUCT_MODEL, build_instruct_prompt, questions, golds, stop=STOP)

Qwen/Qwen2.5-0.5B-Instruct: acc=53.0%  no-answer=23.5%


In [ ]:
save_results(results, STAGE_DIR / "instruct_0shot_val200.json")

Zero-shot with its chat template, `Qwen2.5-0.5B-Instruct` reaches **53%** accuracy on the 200 validation problems, with a **23.5%** no-answer rate.

## General Check

Each model answers 20 fixed non-math prompts (`GENERAL`), judged by hand against a pass rule written before any model was run. Every prompt gets two marks:

- **Content** judges only the first answer against its rule, ignoring anything generated after it.
- **Clean** requires a content pass *and* that the model stopped right after answering.

Keeping them separate isolates two effects: not stopping is what SFT should fix (EOS at the end of each completion), while content is what math specialization may erode. Any output that turns into a math solution or contains a `####` line fails both.

### Base Model

In [ ]:
general_results = generate(BASE_MODEL, lambda q, _: build_prompt(q), [g["prompt"] for g in GENERAL], stop=STOP, max_new_tokens=256)

In [ ]:
save_results(general_results, STAGE_DIR / "base_general.json")

In [ ]:
general_results

[{'output': ' I am sorry, but I cannot attend a meeting at this time. I am currently working on a project and have other commitments. I will let you know if I can attend the meeting.',
  'n_tokens': 38,
  'hit_limit': False},
 {'output': " Dear [Friend's Name],\n\nI hope this message finds you well. I wanted to take a moment to express my sincere gratitude for all the hard work and support you provided during my move. Your unwavering dedication and understanding have been a source of strength and inspiration for me.\n\nYour help was instrumental in ensuring that my new home was a comfortable and welcoming place. From the moment I moved in, you were there to assist me with any questions or concerns I had, and you always made sure to keep me informed about any changes or updates. Your patience and patience were truly appreciated, and I can't thank you enough for that.\n\nI can't wait to see you and to share the joy of moving into your new home. Please know that I will always be grateful 

**Qwen2.5-0.5B base, zero-shot (`Question: … Answer:`)**

| ID | Category | Content | Clean | Note |
| --- | --- | --- | --- | --- |
| w1 | writing | ✗ | ✗ | Extra sentence breaks the 2-sentence rule |
| w2 | writing | ✗ | ✗ | Over the ~80-word limit |
| w3 | writing | ✓ | ✓ | |
| w4 | writing | ✓ | ✓ | |
| f1 | format | ✓ | ✓ | |
| f2 | format | ✓ | ✗ | Repeated the list 3-4 times |
| f3 | format | ✓ | ✓ | |
| f4 | format | ✓ | ✗ | Extra sentence after HELLO |
| t1 | translation | ✓ | ✓ | Correct, slightly unnatural |
| t2 | translation | ✓ | ✗ | Appended an "AI assistant" system prompt |
| k1 | facts | ✓ | ✗ | Appended a "single-select problem" block |
| k2 | facts | ✓ | ✗ | Same as k1 |
| k3 | facts | ✓ | ✗ | Same as k1 |
| k4 | facts | ✓ | ✗ | Same as k1 |
| r1 | rewrite | ✓ | ✗ | Appended an "AI assistant" system prompt |
| r2 | rewrite | ✗ | ✗ | Copied the input unchanged |
| r3 | rewrite | ✓ | ✓ | |
| j1 | judgment | ✓ | ✗ | Appended an "AI assistant" system prompt |
| j2 | judgment | ✗ | ✗ | Wrong answer |
| j3 | judgment | ✓ | ✗ | Appended an "AI assistant" system prompt |
| **Total** | | **16/20** | **6/20** | |

The base model knows most answers but rarely stops. The appended text is instruction-tuning data from pretraining showing through, which suggests the "base" model is not a clean base.

### Instruct Model

In [ ]:
general_results = generate(INSTRUCT_MODEL, build_instruct_prompt, [g["prompt"] for g in GENERAL], stop=STOP, max_new_tokens=256)

In [ ]:
save_results(general_results, STAGE_DIR / "instruct_general.json")

In [ ]:
general_results

[{'output': "Dear [Recipient's Name],\n\nI am writing to decline your invitation to our upcoming meeting. I believe that the time and resources you have allocated for this meeting are better utilized elsewhere, and I am unable to attend.\n\nThank you for considering my suggestion.\n\n#### <number>",
  'n_tokens': 55,
  'hit_limit': False},
 {'output': 'Certainly! Here\'s a step-by-step guide to writing a thank-you note to your friend who helped you move house:\n\n1. **Start with a warm greeting**: Begin with a warm and friendly greeting, such as "Dear [Friend\'s Name]," to show appreciation for their help.\n\n2. **Express your gratitude**: Express your gratitude for their assistance, acknowledging that they were instrumental in your move. You can say something like, "Thank you so much for your help in moving your house! It was a huge help and I couldn\'t have done it without you."\n\n3. **Provide specific details**: Mention the specific tasks or services you were helped with, such as m

**Qwen2.5-0.5B-Instruct, chat template, zero-shot**

| ID | Category | Content | Clean | Note |
| --- | --- | --- | --- | --- |
| w1 | writing | ✓ | ✓ | |
| w2 | writing | ✗ | ✗ | Wrote a guide on how to write the note instead of the note |
| w3 | writing | ✓ | ✓ | Long |
| w4 | writing | ✓ | ✓ | |
| f1 | format | ✓ | ✓ | Extra boilerplate around valid JSON |
| f2 | format | ✓ | ✓ | Walked through the days, then gave the correct final list |
| f3 | format | ✗ | ✗ | Added a justification; rule allows a word or two |
| f4 | format | ✓ | ✓ | |
| t1 | translation | ✗ | ✗ | Wrong translation |
| t2 | translation | ✓ | ✓ | Explained part by part before the full sentence |
| k1 | facts | ✓ | ✗ | Repeated the answer several times |
| k2 | facts | ✓ | ✓ | Step-by-step ("To determine…"), then the answer |
| k3 | facts | ✓ | ✓ | Same as k2 |
| k4 | facts | ✓ | ✓ | Same as k2 |
| r1 | rewrite | ✓ | ✓ | |
| r2 | rewrite | ✓ | ✓ | |
| r3 | rewrite | ✓ | ✓ | Step-by-step before the fix |
| j1 | judgment | ✓ | ✓ | Step-by-step |
| j2 | judgment | ✓ | ✓ | Step-by-step |
| j3 | judgment | ✓ | ✓ | |
| **Total** | | **17/20** | **16/20** | |

The instruct model stops reliably, which is where it differs from the base. Content is about the same. It applies step-by-step reasoning even to trivia ("To determine…"), a habit its post-training carried into every kind of task.

## Stage 1 Summary

Validation subset, 200 problems (±3.5 pts).

| Model | Prompt | Accuracy | No-answer | General: content | General: clean |
| --- | --- | --- | --- | --- | --- |
| Base | zero-shot | 0% (48.5% lenient) | 96% | 16/20 | 6/20 |
| Base | 8-shot | 48% | 4% | - | - |
| Instruct | chat, zero-shot | 53% | 23.5% | 17/20 | 16/20 |

All three setups solve roughly the same share of problems, about 48-53%, within noise. What changes is behavior: 8 examples teach the base model the format, and Qwen's post-training teaches it to follow instructions, stop, and reason step by step. The knowledge comes from pretraining. Post-training and prompting decide whether it shows up in a usable form.

**Prediction for Stage 2:** SFT should bring zero-shot accuracy to about 48% by fixing format and stopping. A clearly higher score would mean SFT adds more than format.